<a href="https://colab.research.google.com/github/felipesayegg/cnn-gato-cachorro/blob/main/RN_AULA_10_CNN_IMAGENS_COLORIDAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐶🐱 CNN para Classificação de Imagens de Gatos e Cachorros

Este notebook apresenta um projeto prático utilizando Redes Neurais Convolucionais (CNNs) para classificar imagens coloridas entre **gatos** e **cachorros**. Esse é um exemplo clássico de aplicação de visão computacional com **classificação binária**, onde o modelo precisa aprender a identificar padrões visuais para distinguir entre as duas classes.

📌 **Objetivo**: Treinar um modelo de rede neural convolucional (CNN) capaz de receber uma imagem colorida e prever se ela representa um gato ou um cachorro.

🔍 **O que será abordado**:
1. Importação das bibliotecas necessárias  
2. Carregamento e visualização dos dados  
3. Pré-processamento das imagens  
4. Construção do modelo CNN  
5. Treinamento do modelo  
6. Avaliação de desempenho  
7. Conclusão e aprendizados

🧠 **Por que CNN?**  
As CNNs são projetadas para reconhecer padrões espaciais em imagens (bordas, cores, formas), sendo altamente eficazes em tarefas de classificação visual como essa.


In [57]:
# 👇 Etapa opcional: apenas use se estiver tendo problemas com o TensorFlow ou rodando em ambiente local.

# 🔄 Desinstala o TensorFlow atual (caso esteja com bugs ou muito desatualado)
!pip uninstall -y tensorflow

# ⬆️ Atualiza o gerenciador de pacotes pip para garantir que tudo funcione com as versões mais recentes
!pip install --upgrade pip

# 📥 Instala a versão mais atual e estável do TensorFlow
!pip install tensorflow


Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
  Using cached tensorflow-2.19.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached tensorflow-2.19.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (644.9 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.19.0 which is incompatible.


In [1]:

# 📦 Instalando o TensorFlow (uma das bibliotecas mais importantes para Deep Learning)
# A parte '==2.16.1' (se for usada) serve para garantir que estamos usando exatamente a versão 2.16.1,
# que pode ser a mesma usada pela professora ou mais estável para o projeto.
!pip install tensorflow  # ou use !pip install tensorflow==2.16.1 para uma versão específica


In [2]:
# 📐 NumPy é uma biblioteca poderosa para trabalhar com arrays e matrizes numéricas em Python.
# No contexto de redes neurais, usamos muito para manipular os dados de entrada e saída.
import numpy as np

# 🧠 TensorFlow é o coração do nosso projeto — é a biblioteca que vamos usar para construir, treinar e avaliar nossa CNN.
# Ele fornece todas as ferramentas necessárias para trabalhar com redes neurais de forma eficiente.
import tensorflow as tf


In [3]:
# 🔍 Verificando as versões instaladas das bibliotecas
# Isso é importante para garantir que estamos usando versões compatíveis com o código do projeto.

np.__version__, tf.__version__  # Retorna as versões atuais do NumPy e do TensorFlow


('2.0.2', '2.19.0')

In [4]:
import tempfile  # 🔧 Biblioteca do Python que permite criar arquivos e pastas temporários
import zipfile   # 📦 Usada para lidar com arquivos compactados (.zip)

# 📁 Criando um diretório temporário para armazenar os dados (imagens) durante a execução do notebook.
# Esse diretório some automaticamente depois que o código termina, ou seja, é ideal para testes e demonstrações.
temp_dir = tempfile.TemporaryDirectory()

# 🔍 Vamos ver o caminho onde esse diretório temporário foi criado
print(temp_dir)


<TemporaryDirectory '/tmp/tmp22ffrl9z'>


In [5]:
# 📦 Abrindo o arquivo compactado chamado 'dataset.zip' em modo leitura ('r')
# Esse arquivo deve conter as imagens que vamos usar (ex: fotos de gatos e cachorros)
with zipfile.ZipFile('dataset.zip', 'r') as zip:

  # 📂 Extraindo todo o conteúdo do zip para dentro do diretório temporário que criamos antes
  zip.extractall(temp_dir.name)


In [6]:
# 🔨 Estamos importando os "blocos de montar" da nossa rede neural

from tensorflow.keras.models import Sequential
# 📚 Modelo sequencial: permite empilhar camadas de forma linear (uma após a outra)

from tensorflow.keras.layers import (
    InputLayer,         # 🎯 Define o formato da entrada da rede (ex: tamanho das imagens)
    Conv2D,             # 🧩 Camada convolucional: extrai padrões visuais como bordas e formas
    MaxPooling2D,       # 🔽 Reduz a dimensão da imagem (ajuda a evitar overfitting e economiza recursos)
    Flatten,            # 🧷 Achata os dados (de matriz para vetor) antes de passar para camadas densas
    Dense,              # 🔌 Camada totalmente conectada (neurônios clássicos)
    Dropout,            # 💧 Desativa aleatoriamente alguns neurônios (evita overfitting)
    BatchNormalization  # 🧼 Normaliza as ativações para melhorar o desempenho e a estabilidade do treino
)

# 🖼️ Leitura de imagens individuais (caso você precise carregar uma imagem manualmente)
from tensorflow.keras.preprocessing import image

# 🗂️ Leitura automática de imagens a partir de pastas (usado para treino/validação com diretórios organizados)
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [7]:
# 🧠 Criando a CNN com o modelo sequencial — camada por camada
classificador = Sequential()

# 🎯 Camada de entrada: define o tamanho esperado das imagens (64x64 pixels e 3 canais de cor – RGB)
classificador.add(InputLayer(shape = (64, 64, 3)))

# 🔍 1ª camada convolucional: aplica 32 filtros 3x3 na imagem para detectar padrões simples como bordas
classificador.add(Conv2D(32, (3, 3), activation='relu'))

# 🧼 Normaliza a saída da convolução para acelerar o treino e deixar o modelo mais estável
classificador.add(BatchNormalization())

# 🔽 Reduz a dimensão da imagem (metade), mantendo só as informações mais relevantes
classificador.add(MaxPooling2D(pool_size=(2, 2)))

# 🔍 2ª camada convolucional: mais filtros para detectar padrões mais complexos (como orelhas ou focinhos)
classificador.add(Conv2D(32, (3, 3), activation='relu'))
classificador.add(BatchNormalization())
classificador.add(MaxPooling2D(pool_size=(2, 2)))

# 🧷 Transforma a imagem (matriz) em um vetor linear para ser usado nas camadas densas
classificador.add(Flatten())

# 🔌 Primeira camada densa: 128 neurônios totalmente conectados, aprende padrões mais abstratos
classificador.add(Dense(units=128, activation='relu'))

# 💧 Dropout desativa 20% dos neurônios dessa camada durante o treino — isso ajuda a evitar overfitting
classificador.add(Dropout(0.2))

# 🔌 Segunda camada densa: repete o processo para fortalecer o aprendizado
classificador.add(Dense(units=128, activation='relu'))
classificador.add(Dropout(0.2))

# 🎯 Camada de saída com 1 único neurônio (porque é classificação binária: gato ou cachorro)
# A ativação 'sigmoid' retorna um valor entre 0 e 1, representando a probabilidade da imagem ser uma das classes
classificador.add(Dense(units = 1, activation='sigmoid'))


In [8]:
# 📊 Mostra um resumo completo da arquitetura do modelo
# Inclui o tipo de cada camada, as dimensões da saída e a quantidade de parâmetros que serão treinados
classificador.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 62, 62, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 29, 29, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       802,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 829,985 (3.17 MB)

 Trainable params: 829,857 (3.17 MB)

 Non-trainable params: 128 (512.00 B)

In [9]:
# 🛠️ Compilando o modelo: aqui definimos como o modelo será treinado

classificador.compile(
    optimizer='adam',                 # 🚀 Otimizador que ajusta os pesos da rede. O 'adam' é eficiente e funciona bem na maioria dos casos.
    loss='binary_crossentropy',      # ⚖️ Função de perda usada para problemas de classificação binária (gato x cachorro).
    metrics=['accuracy']             # 📈 Métrica usada para avaliar o desempenho do modelo. Aqui vamos acompanhar a acurácia.
)


In [10]:
# 🧪 Criando um gerador de imagens para o conjunto de TREINAMENTO
# Ele aplica transformações aleatórias nas imagens para aumentar a diversidade dos dados (data augmentation)

gerador_treinamento = ImageDataGenerator(
    rescale=1./255,            # 🔄 Normaliza os valores dos pixels (de 0–255 para 0–1)
    rotation_range=7,          # 🔄 Rotaciona a imagem em até 7 graus (simula ângulos diferentes)
    horizontal_flip=True,      # ↔️ Inverte horizontalmente (ex: cachorro olhando para o outro lado)
    shear_range=0.2,           # 💫 Aplica distorção nos eixos (cisalhamento) — deixa a imagem inclinada
    height_shift_range=0.07,   # ⬆️⬇️ Move a imagem verticalmente (simula deslocamentos)
    zoom_range=0.2             # 🔍 Aproxima ou afasta a imagem (zoom)
)


In [11]:
# ✅ Criando o gerador de imagens para o conjunto de TESTE/VALIDAÇÃO
# Aqui a gente só normaliza os pixels — nada de rotação, zoom, nem distorção

gerador_teste = ImageDataGenerator(
    rescale=1./255  # 🔄 Converte os valores dos pixels de 0–255 para 0–1
)


In [12]:
# 📂 Carregando as imagens do diretório de TREINAMENTO
# O gerador vai buscar as imagens nas subpastas (ex: /gato e /cachorro),
# redimensionar, aplicar aumentos e entregar lotes (batches) prontos para o modelo treinar.

base_treinamento = gerador_treinamento.flow_from_directory(
    f'{temp_dir.name}/dataset/training_set',  # 📁 Caminho para as pastas de treino (gato e cachorro)
    target_size=(64, 64),                     # 🖼️ Redimensiona todas as imagens para 64x64 pixels
    batch_size=32,                            # 📦 Processa 32 imagens por vez
    class_mode='binary'                       # 🧾 Como temos só duas classes, usamos modo binário
)


Found 4000 images belonging to 2 classes.


In [13]:
# 📂 Carregando as imagens do diretório de TESTE/VALIDAÇÃO
# Aqui as imagens são apenas redimensionadas e normalizadas, sem distorções nem aumentos.

base_teste = gerador_teste.flow_from_directory(
    f'{temp_dir.name}/dataset/test_set',  # 📁 Caminho para as pastas de teste (gato e cachorro)
    target_size=(64, 64),                 # 🖼️ Redimensiona todas as imagens para 64x64 pixels
    batch_size=32,                        # 📦 Lote de 32 imagens por vez
    class_mode='binary'                   # 🧾 Saída binária (0 ou 1) — gato ou cachorro
)


Found 1000 images belonging to 2 classes.


In [14]:
# aumentar o numero de epocas para melhorar a rede
# 🚀 Treinando o modelo com os dados de treino e avaliando com os dados de teste/validação

classificador.fit(
    base_treinamento,              # 📚 Dados de treino (com data augmentation)
    epochs=10,                     # 🔁 Número de vezes que o modelo vai passar por todo o conjunto de treino
    validation_data=base_teste     # 🧪 Dados de validação para avaliar o desempenho ao final de cada época
)


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 18s 99ms/step - accuracy: 0.5750 - loss: 0.9032 - val_accuracy: 0.5680 - val_loss: 0.6847
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 89ms/step - accuracy: 0.6047 - loss: 0.6784 - val_accuracy: 0.5160 - val_loss: 1.4311
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 88ms/step - accuracy: 0.6485 - loss: 0.6491 - val_accuracy: 0.6200 - val_loss: 0.6706
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 87ms/step - accuracy: 0.6698 - loss: 0.6092 - val_accuracy: 0.6110 - val_loss: 0.7332
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 86ms/step - accuracy: 0.6680 - loss: 0.5867 - val_accuracy: 0.7000 - val_loss: 0.5851
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 92ms/step - accuracy: 0.6876 - loss: 0.5796 - val_accuracy: 0.6560 - val_loss: 0.6095
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.7113 - loss: 0.5491 - val_accuracy: 0.6520 - val_loss: 0.6371
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 84ms/step - accuracy: 0.7294 - loss: 0.5296 - 

# PREDICT DE NOVAS IMAGENS

In [16]:
# 🖼️ Carregando uma imagem de teste individual para fazer uma previsão
# Neste exemplo, estamos pegando uma imagem de um cachorro

imagem_teste = image.load_img(
    f'{temp_dir.name}/dataset/test_set/cachorro/dog.3500.jpg',  # 📂 Caminho da imagem
    target_size=(64, 64)  # 🖼️ Redimensiona para 64x64 pixels, igual ao padrão da CNN
)


In [17]:
# 🔍 Verificando o tipo da variável imagem_teste
# Isso ajuda a entender o formato atual da imagem carregada

type(imagem_teste)


PIL.Image.Image

In [18]:
# 🔄 Convertendo a imagem de formato PIL para um array NumPy (matriz de pixels)
# Agora a imagem passa a ser representada por números — que é o que a rede entende

imagem_teste = image.img_to_array(imagem_teste)


In [19]:
imagem_teste

array([[[120., 125., 119.],
        [106., 111., 105.],
        [114., 119., 113.],
        ...,
        [ 58.,  42.,  29.],
        [ 65.,  38.,  29.],
        [ 70.,  35.,  29.]],

       [[105., 110., 104.],
        [129., 134., 128.],
        [134., 139., 133.],
        ...,
        [ 59.,  41.,  29.],
        [ 66.,  42.,  32.],
        [ 69.,  40.,  32.]],

       [[126., 131., 125.],
        [131., 136., 130.],
        [118., 123., 117.],
        ...,
        [ 64.,  40.,  30.],
        [ 64.,  42.,  31.],
        [ 63.,  41.,  30.]],

       ...,

       [[158., 165., 183.],
        [188., 198., 208.],
        [184., 196., 208.],
        ...,
        [ 96.,  90.,  92.],
        [ 90.,  84.,  86.],
        [ 76.,  70.,  72.]],

       [[170., 172., 185.],
        [196., 198., 211.],
        [189., 191., 203.],
        ...,
        [ 96.,  87.,  90.],
        [ 85.,  76.,  79.],
        [ 83.,  74.,  77.]],

       [[176., 179., 184.],
        [185., 188., 193.],
        [184., 1

In [20]:
# 🔧 Normalizando os valores dos pixels
# As imagens originalmente têm valores de 0 a 255 (RGB)
# Dividindo por 255, transformamos todos os valores para ficar entre 0 e 1

imagem_teste /= 255


In [21]:
imagem_teste

array([[[0.47058824, 0.49019608, 0.46666667],
        [0.41568628, 0.43529412, 0.4117647 ],
        [0.44705883, 0.46666667, 0.44313726],
        ...,
        [0.22745098, 0.16470589, 0.11372549],
        [0.25490198, 0.14901961, 0.11372549],
        [0.27450982, 0.13725491, 0.11372549]],

       [[0.4117647 , 0.43137255, 0.40784314],
        [0.5058824 , 0.5254902 , 0.5019608 ],
        [0.5254902 , 0.54509807, 0.52156866],
        ...,
        [0.23137255, 0.16078432, 0.11372549],
        [0.25882354, 0.16470589, 0.1254902 ],
        [0.27058825, 0.15686275, 0.1254902 ]],

       [[0.49411765, 0.5137255 , 0.49019608],
        [0.5137255 , 0.53333336, 0.50980395],
        [0.4627451 , 0.48235294, 0.45882353],
        ...,
        [0.2509804 , 0.15686275, 0.11764706],
        [0.2509804 , 0.16470589, 0.12156863],
        [0.24705882, 0.16078432, 0.11764706]],

       ...,

       [[0.61960787, 0.64705884, 0.7176471 ],
        [0.7372549 , 0.7764706 , 0.8156863 ],
        [0.72156864, 0

In [22]:
imagem_teste.shape

(64, 64, 3)

In [23]:
# ➕ Adicionando uma nova dimensão ao array da imagem
# Isso transforma (64, 64, 3) em (1, 64, 64, 3), simulando um "lote" com 1 imagem

imagem_teste = np.expand_dims(imagem_teste, axis=0)


In [24]:
imagem_teste.shape

(1, 64, 64, 3)

In [25]:
# 🔮 Fazendo a previsão com a imagem processada
# O modelo vai retornar um valor entre 0 e 1, representando a probabilidade da imagem ser da classe 1

previsao = classificador.predict(imagem_teste)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 570ms/step


In [26]:
previsao

array([[0.883357]], dtype=float32)

In [27]:
# ✅ Convertendo a previsão de probabilidade para classe binária
# Se for maior que 0.5, será considerado 1 (ex: cachorro)
# Se for menor ou igual a 0.5, será considerado 0 (ex: gato)

previsao = previsao > 0.5


In [28]:
previsao

array([[ True]])

In [29]:
# 🔍 Mostrando o mapeamento das classes (índice de cada pasta)
# Isso diz qual classe o modelo considera como 0 e qual como 1

base_treinamento.class_indices


{'cachorro': 0, 'gato': 1}

In [33]:
# 🧠 Criando uma função para prever se uma nova imagem é um GATO ou CACHORRO

def predict_image(caminho):
  # 🖼️ Carrega a imagem do caminho informado e redimensiona para 64x64 pixels (tamanho padrão da CNN)
  imagem_teste = image.load_img(caminho, target_size=(64,64))

  # 🔢 Converte a imagem para um array NumPy (matriz de pixels)
  imagem_teste = image.img_to_array(imagem_teste)

  # ⚖️ Normaliza os valores dos pixels para ficar entre 0 e 1 (igual ao que foi feito no treino)
  imagem_teste /= 255

  # ➕ Adiciona uma dimensão extra para simular um "batch" com uma única imagem
  imagem_teste = np.expand_dims(imagem_teste, axis=0)

  # 🔮 Faz a previsão usando o modelo e pega o valor retornado (probabilidade)
  prediction = classificador.predict(imagem_teste)[0]

  # 🧾 Interpreta o resultado: se for menor que 0.5, é cachorro (classe 0); senão, é gato (classe 1)
  return "Cachorro" if prediction < 0.5 else "Gato"



In [34]:
predict_image("/content/cat.15.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


'Gato'

In [35]:
predict_image("/content/cat.8.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


'Gato'

In [38]:
predict_image("/content/dog.11.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


'Cachorro'

# ✅ Conclusão – Projeto CNN para Classificação de Gatos e Cachorros 🐶🐱

Neste projeto, desenvolvemos uma **Rede Neural Convolucional (CNN)** do zero utilizando o TensorFlow/Keras para resolver um problema clássico de **classificação binária de imagens**: identificar se uma imagem colorida representa um **gato** ou um **cachorro**.

---

## 🧠 O que é uma CNN (Convolutional Neural Network)?

As **CNNs** são redes neurais especialmente projetadas para lidar com dados que possuem uma estrutura espacial — como **imagens**.  
Ao invés de receber os pixels "achatados" como uma simples sequência de números, as CNNs processam as imagens como matrizes, preservando a relação entre os pixels vizinhos (como contornos, cores, formas, etc).

Elas funcionam em etapas:

1. **Camadas Convolucionais (Conv2D)** – detectam padrões visuais como bordas, texturas e formas.
2. **Camadas de Pooling** – reduzem a dimensão da imagem, mantendo os padrões mais importantes.
3. **Camadas Densas (Fully Connected)** – fazem a parte "decisora", combinando os padrões para chegar à resposta.
4. **Camada de Saída com Sigmoid** – transforma a saída em uma probabilidade entre 0 e 1.

Esse tipo de arquitetura é **altamente eficaz em tarefas de visão computacional**, como detecção de rostos, análise de raio-X, reconhecimento facial, entre muitas outras.

---

## ⚙️ O que foi feito neste projeto:

- Utilizamos o `ImageDataGenerator` para carregar e processar automaticamente imagens organizadas em diretórios.
- Aplicamos **data augmentation** para simular diferentes variações das imagens e evitar overfitting.
- Construímos uma CNN com duas camadas convolucionais e camadas densas para aprender os padrões visuais.
- Treinamos o modelo por várias épocas, avaliando seu desempenho com métricas como **acurácia** e **loss**.
- Criamos uma **função reutilizável** para classificar novas imagens de forma prática e automatizada.

---

## 🎯 Resultado:

O modelo treinado demonstrou a capacidade de distinguir com boa precisão entre **gatos e cachorros** em imagens nunca vistas. A cada época, o modelo ajustou seus pesos para melhorar sua capacidade de generalização.

A arquitetura usada pode ser expandida para problemas mais complexos, como:
- Detecção de múltiplas classes (mais de dois tipos de animais, por exemplo)
- Uso de modelos pré-treinados (Transfer Learning)
- Classificação de imagens reais em ambientes de produção

---

## 🚀 Considerações finais

Este projeto foi uma excelente introdução prática ao uso de **Redes Neurais Convolucionais** com Python e TensorFlow. Ele mostra como a IA pode ser aplicada a problemas visuais com dados reais e prepara o terreno para projetos mais avançados no campo de Visão Computacional.

---

### 🔗 Conecte-se comigo!

Se você gostou deste projeto ou quer trocar ideias sobre inteligência artificial, deep learning ou projetos com impacto real, será um prazer conectar com você aqui no LinkedIn! 💬
